In [2]:
import pandas as pd

def generate_submission():
    df = pd.read_csv("filled_dataset.csv")
    orig = pd.read_csv("dataset.csv")
    train = pd.read_csv("train_dataset.csv")
    
    # Force string conversion to prevent subtle datatype mismatches
    df['kaggle_id'] = df['datetime'].astype(str) + "||" + df['ticker'].astype(str)
    id_map = dict(zip(df['kaggle_id'], df['final_value']))
    
    rows = []
    feature_cols = [c for c in orig.columns if c not in ["datetime", "underlying_price"]]
    
    # Use training data for the median to prevent NaN contamination
    global_median = train['implied_volatility'].median()
    
    for idx, row in orig.iterrows():
        for col in feature_cols:
            if pd.isna(row[col]):
                uid = f"{str(row['datetime'])}||{str(col)}"
                
                # Fetch mapped calculation
                val = id_map.get(uid, global_median)
                
                # Ultimate Safety Check: If the mapped value is somehow NaN, clamp it
                if pd.isna(val):
                    val = global_median
                    
                rows.append({"id": uid, "value": val})
                
    submission = pd.DataFrame(rows)
    submission.to_csv("submission.csv", index=False)
    
    print(f"Process complete: submission.csv generated with {len(submission)} valid rows.")

generate_submission()

Process complete: submission.csv generated with 5460 valid rows.
